# Gold - longevity metrics

Builds the Gold tables behind the **Longevity** report page from the source-agnostic Silver model.
Device-measured values (Apple Health, Garmin, Oura, ...) always win; until such a source is connected the
metrics are estimated from Strava workouts and the `method` / `is_estimate` columns say so.

| Gold table | Grain | Content |
|---|---|---|
| `FactDailyActivity` | calendar day (every day, also rest days) | Workouts, active / moderate / vigorous / Zone 2 minutes, WHO-equivalent minutes, strength & cardio sessions, steps, distance, energy, load |
| `FactBiometric` | day + metric + method | VO2max, FTP, W/kg, body weight, max / resting HR, HRV - measured or estimated |
| `DimPeriod` | report period + day | Bridge behind the Period slicer: Last 4 weeks, 3 / 6 / 12 months, This year, Last 2 years |

**Estimation rules**

| Metric | Rule |
|---|---|
| Moderate / vigorous minutes | Heart-rate zone time (Z1-Z2 = moderate, Z3-Z5 = vigorous); without zones: avg HR >= 77 % of max HR = vigorous; strength / mobility = moderate |
| Zone 2 minutes | Time in HR zone 2; without zones the whole workout when avg HR is 60-72 % of max HR |
| WHO-equivalent minutes | moderate + 2 x vigorous (WHO guideline: 150-300 min / week) |
| Steps | Device `steps_count` when available, otherwise running / walking workouts: cadence x 2 x minutes, or distance / stride length |
| VO2max (cycling) | ACSM: 10.8 x best 5-min power (W) / body weight (kg) + 7 |
| VO2max (running) | Jack Daniels VDOT from best-effort distance and time (1.5 km - marathon) |
| FTP W/kg | FTP / body weight on the same day (latest known weight) |

In [ ]:
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import json

spark.conf.set("spark.sql.parquet.vorder.default", "true")

WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]
def lakehouse_id(name):
    """Resolve a lakehouse in this workspace by name, so the notebook works in any workspace."""
    try:
        item = notebookutils.lakehouse.get(name)
        return item["id"] if isinstance(item, dict) else item.id
    except Exception:
        import sempy.fabric as fabric
        return fabric.resolve_item_id(name, type="Lakehouse")


SILVER_LAKEHOUSE_ID = lakehouse_id("Strava_Silver")
SILVER_TABLES = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SILVER_LAKEHOUSE_ID}/Tables"

ACTIVE_DAY_MINUTES = 30
VIGOROUS_HR_RATIO = 0.77
ZONE2_HR_RATIO = (0.60, 0.72)
RUN_STRIDE_M, WALK_STRIDE_M = 1.10, 0.75
CARDIO_FAMILIES = ["cycling", "running", "swimming", "rowing", "conditioning", "walking"]
STRENGTH_FAMILIES = ["strength"]
LOW_INTENSITY_FAMILIES = ["strength", "mobility"]


def read_silver(name):
    return spark.read.format("delta").load(f"{SILVER_TABLES}/{name}")


def save_gold(df, name):
    df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(f"Tables/{name}")

In [ ]:
# ---------------------------------------------------------------- Workout-level intensity minutes and steps
workouts = read_silver("Workout").where("is_primary_record")
user_max_hr = (
    workouts.where(F.col("max_heart_rate_bpm").between(120, 230)).agg(F.max("max_heart_rate_bpm")).collect()[0][0]
) or 190.0

zones = (
    read_silver("WorkoutZoneTime").where(F.col("zone_type") == "heart_rate")
    .groupBy("workout_id")
    .agg(
        F.sum("seconds_in_zone").alias("z_total_s"),
        F.sum(F.when(F.col("zone_number") <= 2, F.col("seconds_in_zone")).otherwise(0)).alias("z12_s"),
        F.sum(F.when(F.col("zone_number") == 2, F.col("seconds_in_zone")).otherwise(0)).alias("z2_s"),
        F.sum(F.when(F.col("zone_number") >= 3, F.col("seconds_in_zone")).otherwise(0)).alias("z345_s"),
    )
)

minutes = F.coalesce(F.col("moving_duration_s"), F.col("elapsed_duration_s"), F.lit(0)) / 60.0
hr_ratio = F.col("avg_heart_rate_bpm") / F.lit(float(user_max_hr))
has_zones = F.col("z_total_s") > 0
family = F.col("activity_family")

wk = (
    workouts.join(zones, "workout_id", "left")
    .withColumn("minutes", minutes)
    .withColumn(
        "vigorous_minutes",
        F.when(family.isin(LOW_INTENSITY_FAMILIES), F.lit(0.0))
         .when(has_zones, F.col("minutes") * F.col("z345_s") / F.col("z_total_s"))
         .when(F.col("avg_heart_rate_bpm").isNotNull(), F.when(hr_ratio >= VIGOROUS_HR_RATIO, F.col("minutes")).otherwise(0.0))
         .when(family.isin("running", "rowing", "conditioning"), F.col("minutes"))
         .otherwise(F.lit(0.0)),
    )
    .withColumn("moderate_minutes", F.col("minutes") - F.col("vigorous_minutes"))
    .withColumn(
        "zone2_minutes",
        F.when(family.isin(LOW_INTENSITY_FAMILIES), F.lit(0.0))
         .when(has_zones, F.col("minutes") * F.col("z2_s") / F.col("z_total_s"))
         .when(hr_ratio.between(*ZONE2_HR_RATIO), F.col("minutes"))
         .otherwise(F.lit(0.0)),
    )
    .withColumn(
        "workout_steps",
        F.when(~family.isin("running", "walking"), F.lit(0.0))
         .when(F.col("avg_cadence") > 0,
               F.col("avg_cadence") * F.when(F.col("avg_cadence") < 130, F.lit(2.0)).otherwise(F.lit(1.0)) * F.col("minutes"))
         .when(F.col("distance_m") > 0,
               F.col("distance_m") / F.when(family == "running", F.lit(RUN_STRIDE_M)).otherwise(F.lit(WALK_STRIDE_M)))
         .otherwise(F.lit(0.0)),
    )
)

workout_day = (
    wk.groupBy(F.col("local_date").alias("date_key"))
    .agg(
        F.count("*").cast("int").alias("workouts"),
        F.sum(F.when(family.isin(CARDIO_FAMILIES), 1).otherwise(0)).cast("int").alias("cardio_sessions"),
        F.sum(F.when(family.isin(STRENGTH_FAMILIES), 1).otherwise(0)).cast("int").alias("strength_sessions"),
        F.round(F.sum("minutes"), 1).alias("active_minutes"),
        F.round(F.sum("moderate_minutes"), 1).alias("moderate_minutes"),
        F.round(F.sum("vigorous_minutes"), 1).alias("vigorous_minutes"),
        F.round(F.sum("zone2_minutes"), 1).alias("zone2_minutes"),
        F.round(F.sum("workout_steps"), 0).alias("workout_steps"),
        F.round(F.sum(F.coalesce("distance_m", F.lit(0.0))) / 1000.0, 2).alias("distance_km"),
        F.round(F.sum(F.coalesce("active_energy_kcal", F.lit(0.0))), 0).alias("active_energy_kcal"),
        F.round(F.sum(F.coalesce("training_load_score", F.lit(0.0))), 1).alias("training_load"),
        F.concat_ws(", ", F.array_sort(F.collect_set(F.initcap(F.regexp_replace("activity_family", "_", " "))))).alias("sports"),
    )
)

# ---------------------------------------------------------------- Device daily totals (Apple Health / Garmin, when connected)
observations = read_silver("HealthObservation")
device_steps = (
    observations.where(F.col("metric_code") == "steps_count")
    .groupBy("local_date", "source_system").agg(F.max("value").alias("steps"))
    .groupBy(F.col("local_date").alias("date_key")).agg(F.max("steps").alias("device_steps"))
)

first_day = workouts.agg(F.min("local_date")).collect()[0][0] or date.today()
calendar = spark.sql(f"SELECT explode(sequence(to_date('{first_day}'), current_date())) AS date_key")

daily = (
    calendar.join(workout_day, "date_key", "left").join(device_steps, "date_key", "left")
    .fillna(0, subset=["workouts", "cardio_sessions", "strength_sessions", "active_minutes", "moderate_minutes",
                       "vigorous_minutes", "zone2_minutes", "workout_steps", "distance_km", "active_energy_kcal", "training_load"])
    .withColumn("who_minutes", F.round(F.col("moderate_minutes") + 2 * F.col("vigorous_minutes"), 1))
    .withColumn("steps", F.coalesce("device_steps", "workout_steps").cast("long"))
    .withColumn("steps_method", F.when(F.col("device_steps").isNotNull(), "device").otherwise("estimated_from_workouts"))
    .withColumn("is_active_day", F.col("active_minutes") >= ACTIVE_DAY_MINUTES)
    .withColumn("is_rest_day", F.col("workouts") == 0)
    .withColumn("sports", F.coalesce("sports", F.lit("Rest")))
    .withColumn("active_minutes_7d", F.round(F.sum("active_minutes").over(Window.orderBy("date_key").rowsBetween(-6, 0)), 1))
    .drop("device_steps")
)
save_gold(daily, "FactDailyActivity")

In [ ]:
# ---------------------------------------------------------------- Biometrics: measured + estimated (long format)
metric_names = read_silver("HealthMetric").select("metric_code", "metric_name", F.col("canonical_unit").alias("unit"))

measured = observations.where(F.col("metric_code") != "steps_count").select(
    F.col("local_date").alias("date_key"), "metric_code", "value", "source_system",
    F.col("observation_method").alias("method"),
    F.col("observation_method").startswith("estimated").alias("is_estimate"),
)

weights = (
    observations.where((F.col("metric_code") == "body_weight_kg") & (F.col("value").between(30, 250)))
    .groupBy("local_date").agg(F.avg("value").alias("weight_kg"))
)
latest_weight = weights.orderBy(F.desc("local_date")).limit(1).collect()
fallback_weight = float(latest_weight[0]["weight_kg"]) if latest_weight else None


def with_weight(df):
    """Attach the most recent body weight known on or before each row's date (falls back to latest weight)."""
    if fallback_weight is None:
        return df.withColumn("weight_kg", F.lit(None).cast("double"))
    w = Window.partitionBy("_row").orderBy(F.desc("local_date"))
    joined = (
        df.withColumn("_row", F.monotonically_increasing_id())
        .join(weights, F.col("local_date") <= F.col("date_key"), "left")
        .withColumn("_r", F.row_number().over(w)).where("_r = 1")
    )
    return joined.withColumn("weight_kg", F.coalesce("weight_kg", F.lit(fallback_weight))).drop("_row", "_r", "local_date")


efforts = (
    read_silver("WorkoutBestEffort").select("workout_id", "effort_type", "effort_name", "duration_s", "distance_m", "avg_power_w")
    .join(workouts.select("workout_id", "local_date", "activity_family", "source_system"), "workout_id")
)

# Cycling VO2max (ACSM leg-ergometry): 10.8 * W/kg + 7 using best 5-minute power of each ride
cycling_vo2 = with_weight(
    efforts.where((F.col("activity_family") == "cycling") & (F.col("effort_type") == "power_duration")
                  & (F.col("duration_s") == 300) & F.col("avg_power_w").between(50, 800))
    .groupBy(F.col("local_date").alias("date_key"), "source_system").agg(F.max("avg_power_w").alias("p5"))
).where(F.col("weight_kg").isNotNull()).select(
    "date_key", F.lit("vo2max_ml_kg_min").alias("metric_code"),
    F.round(10.8 * F.col("p5") / F.col("weight_kg") + 7, 1).alias("value"), "source_system",
    F.lit("estimated_cycling_5min_power_acsm").alias("method"), F.lit(True).alias("is_estimate"),
)

# Running VO2max (Daniels VDOT) from best-effort distance + time
t_min = F.col("duration_s") / 60.0
v = F.col("distance_m") / t_min
vo2 = -4.60 + 0.182258 * v + 0.000104 * v * v
pct_max = 0.8 + 0.1894393 * F.exp(-0.012778 * t_min) + 0.2989558 * F.exp(-0.1932605 * t_min)
running_vo2 = (
    efforts.where((F.col("activity_family") == "running") & (F.col("effort_type") == "distance")
                  & F.col("distance_m").between(1500, 42300) & (F.col("duration_s") > 0))
    .withColumn("vdot", vo2 / pct_max)
    .where(F.col("vdot").between(20, 90))
    .groupBy(F.col("local_date").alias("date_key"), "source_system").agg(F.max("vdot").alias("vdot"))
    .select("date_key", F.lit("vo2max_ml_kg_min").alias("metric_code"), F.round("vdot", 1).alias("value"), "source_system",
            F.lit("estimated_running_vdot").alias("method"), F.lit(True).alias("is_estimate"))
)

ftp = measured.where(F.col("metric_code") == "ftp_watts")
ftp_per_kg = with_weight(ftp).where(F.col("weight_kg").isNotNull()).select(
    "date_key", F.lit("ftp_w_per_kg").alias("metric_code"), F.round(F.col("value") / F.col("weight_kg"), 2).alias("value"),
    "source_system", F.concat(F.col("method"), F.lit("_per_kg")).alias("method"), "is_estimate",
)

custom_names = spark.createDataFrame(
    [("ftp_w_per_kg", "FTP per kg", "W/kg")], "metric_code string, metric_name string, unit string")
biometrics = (
    measured.unionByName(cycling_vo2).unionByName(running_vo2).unionByName(ftp_per_kg)
    .where(F.col("value").isNotNull() & F.col("date_key").isNotNull())
    .join(metric_names.unionByName(custom_names), "metric_code", "left")
    .withColumn("metric_name", F.coalesce("metric_name", F.initcap(F.regexp_replace("metric_code", "_", " "))))
    .withColumn("method_label", F.when(F.col("method") == "estimated_cycling_5min_power_acsm", "Estimated from cycling power")
                .when(F.col("method") == "estimated_running_vdot", "Estimated from running efforts")
                .when(F.col("is_estimate"), "Estimated")
                .when(F.col("method").contains("self_reported"), "Profile")
                .otherwise("Measured"))
    .select("date_key", "metric_code", "metric_name", "unit", "value", "method", "method_label", "is_estimate", "source_system")
)
save_gold(biometrics, "FactBiometric")

In [ ]:
# ---------------------------------------------------------------- Report periods (bridge behind the single-select Period slicer)
# One row per (period, day), relative to today, so the daily refresh keeps "Last 3 months" etc. current.
# The report has no "all time" option on purpose: charts never span more than two years.
today = F.current_date()
history_start = F.lit(first_day).cast("date")
period_starts = [
    (1, "Last 4 weeks", F.date_sub(today, 27)),
    (2, "Last 3 months", F.date_add(F.add_months(today, -3), 1)),
    (3, "Last 6 months", F.date_add(F.add_months(today, -6), 1)),
    (4, "This year", F.trunc(today, "year")),
    (5, "Last 12 months", F.date_add(F.add_months(today, -12), 1)),
    (6, "Last 2 years", F.date_add(F.add_months(today, -24), 1)),
]
DEFAULT_PERIOD = "Last 6 months"
dim_period = None
for sort_key, label, start in period_starts:
    rows = spark.range(1).select(
        F.lit(sort_key).alias("period_sort"), F.lit(label).alias("period_label"),
        F.explode(F.sequence(F.greatest(start, history_start), today)).alias("date_key"),
    )
    dim_period = rows if dim_period is None else dim_period.unionByName(rows)
dim_period = dim_period.withColumn("is_default", F.col("period_label") == DEFAULT_PERIOD)
save_gold(dim_period, "DimPeriod")

In [ ]:
summary = {
    "status": "success",
    "layer": "gold",
    "daily_rows": daily.count(),
    "biometric_rows": biometrics.count(),
    "period_rows": dim_period.count(),
    "biometric_metrics": sorted(r[0] for r in biometrics.select("metric_code").distinct().collect()),
    "user_max_hr": user_max_hr,
    "has_body_weight": fallback_weight is not None,
}
print(json.dumps(summary, indent=2, default=str))
notebookutils.notebook.exit(json.dumps(summary, default=str))